In [3]:
import pandas as pd
df = pd.read_csv("./data.csv", index_col=0)
print(df['Month'])

0         1
1         1
2         1
3         1
4         1
         ..
52950    11
52951    11
52952    10
52953    11
52954    12
Name: Month, Length: 52955, dtype: int64


In [4]:
month_abbr = {1: 'Jan',2: 'Fév',3: 'Mar',4: 'Avr', 5: 'Mai',6: 'Juin',7: 'Juil',8: 'Août',9: 'Sep',10: 'Oct',11: 'Nov',12: 'Déc'}
month_name = {1: 'Janvier',2: 'Février',3: 'Mars', 4: 'Avril', 5: 'Mai',6: 'Juin',7: 'Juillet',8: 'Août',9: 'Septembre',10: 'Octobre',11: 'Novembre',12: 'Décembre'}
def indicateur_du_mois(data, current_month = 12, freq=True, abbr=False): 
    previous_month = current_month - 1 if current_month > 1 else 12
    if freq : 
        resultat = data['Month'][(data['Month'] == current_month) | (data['Month'] == previous_month)].value_counts()
        resultat = resultat.sort_index()
        resultat.index = [(month_abbr[i] if abbr else month_name[i]) for i in resultat.index]
        return resultat
    else:
        resultat = data[(data['Month'] == current_month) | (data['Month'] == previous_month)].groupby('Month').apply(calculer_chiffre_affaire)
        resultat.index = [(month_abbr[i] if abbr else month_name[i]) for i in resultat.index]
        return resultat
indicateur_du_mois(df)

Novembre    3967
Décembre    4506
Name: Month, dtype: int64

In [2]:
import dash
import dash_table
from dash import html, dcc, Input, Output, callback
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

df = pd.read_csv("./data.csv", index_col=0)
df = df[['CustomerID', 'Gender', 'Location', 'Product_Category', 'Quantity', 'Avg_Price', 'Transaction_Date', 'Month', 'Discount_pct']]
df['CustomerID'] = df['CustomerID'].fillna(0).astype(int)
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Total_price'] = df['Quantity'] * df['Avg_Price'] * (1 - (df['Discount_pct'] / 100)).round(3)

def calculer_chiffre_affaire(data):
    return data['Total_price'].sum()

def frequence_meilleure_vente(data, top=10, ascending=False):
    resultat = pd.crosstab(
        [data['Gender'], data['Product_Category']], 
        'Total vente', 
        values=data['Total_price'], 
        aggfunc= lambda x : len(x), 
        rownames=['Sexe', 'Categorie du produit'],
        colnames=['']
    ).reset_index().groupby(
        ['Sexe'], as_index=False, group_keys=True
    ).apply(
        lambda x: x.sort_values('Total vente', ascending=ascending).iloc[:top, :]
    ).reset_index(drop=True).set_index(['Sexe', 'Categorie du produit'])

    return resultat

month_abbr = {1: 'Jan',2: 'Fév',3: 'Mar',4: 'Avr', 5: 'Mai',6: 'Juin',7: 'Juil',8: 'Août',9: 'Sep',10: 'Oct',11: 'Nov',12: 'Déc'}
month_name = {1: 'Janvier',2: 'Février',3: 'Mars', 4: 'Avril', 5: 'Mai',6: 'Juin',7: 'Juillet',8: 'Août',9: 'Septembre',10: 'Octobre',11: 'Novembre',12: 'Décembre'}
def indicateur_du_mois(data, current_month = 12, freq=True, abbr=False): 
    previous_month = current_month - 1 if current_month > 1 else 12
    if freq : 
        resultat = data['Month'][(data['Month'] == current_month) | (data['Month'] == previous_month)].value_counts()
        resultat = resultat.sort_index()
        resultat.index = [(month_abbr[i] if abbr else month_name[i]) for i in resultat.index]
        return resultat
    else:
        resultat = data[(data['Month'] == current_month) | (data['Month'] == previous_month)].groupby('Month').apply(calculer_chiffre_affaire)
        resultat.index = [(month_abbr[i] if abbr else month_name[i]) for i in resultat.index]
        return resultat

def barplot_top_10_ventes(data) :
    df_plot = frequence_meilleure_vente(data, ascending=True)
    graph = px.bar(
        df_plot,
        x='Total vente', 
        y=df_plot.index.get_level_values(1),
        color=df_plot.index.get_level_values(0), 
        barmode='group',
        title="Frequence des 10 meilleures ventes",
        labels={"x": "Fréquence", "y": "Categorie du produit", "color": "Sexe"},
        width=680, height=600
    ).update_layout(
        margin = dict(t=60)
    )
    return graph

# Evolution chiffre d'affaire
def plot_evolution_chiffre_affaire(data) :
    df_plot = data.groupby(pd.Grouper(key='Transaction_Date', freq='W')).apply(calculer_chiffre_affaire)[:-1]
    chiffre_evolution = px.line(
        x=df_plot.index, y=df_plot,
        title="Evolution du chiffre d'affaire par semaine",
        labels={"x": "Semaine", "y": "Chiffre d'affaire"},
    ).update_layout( 
        width=800, height=400,
        margin=dict(r = 100, t=60, b=0),
        
    )
    return chiffre_evolution

# Chiffre d'affaire du mois
def plot_chiffre_affaire_mois(data) :
    df_plot = indicateur_du_mois(data, freq=False)
    indicateur = go.Figure(
        go.Indicator(
            mode = "number+delta",
            value = df_plot[1],
            delta = {'reference': df_plot[0]},
            domain = {'row': 0, 'column': 1},
            title=f"{df_plot.index[1]}",
        )
    ).update_layout(
        width=150, height=150, 
        margin=dict(l=20, r=10, t=20, b=0)
    )
    return indicateur

# Ventes du mois
def plot_vente_mois(data, abbr=False) :
    df_plot = indicateur_du_mois(data, freq=True, abbr=abbr)
    indicateur = go.Figure(
        go.Indicator(
            mode = "number+delta",
            value = df_plot[1],
            delta = {'reference': df_plot[0]},
            domain = {'row': 0, 'column': 1},
            title=f"{df_plot.index[1]}",
        )
    ).update_layout( 
        width=150, height=150, 
        margin=dict(l=20, r=10, t=20, b=0)
    )
    return indicateur

# Table
def table_data(df) :
    columns_to_display = ['CustomerID', 'Gender', 'Location', 'Product_Category', 'Quantity', 'Avg_Price', 'Transaction_Date', 'Month', 'Discount_pct']
    df_last_100_sales = base.sort_values(by='Transaction_Date', ascending=False).head(100)
    table_last_100_sales = html.Div([
            html.H3("Table des 100 derniéres ventes"),
            dash_table.DataTable(
            id='fig-5',
            columns=[{"name": i, "id": i} for i in columns_to_display],
            base=df_last_100_sales[columns_to_display].to_dict('records'),
            page_size=10
        )
    ])
    return table_last_100_sales



app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

colors = {
    'background': '#95b4c9',
    'texte': '#4c524d'
}

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col([html.Span("ECAP Store",style={'font-weight': 'bold', 'font-size': 'larger'})]),
        dbc.Col([
            dcc.Dropdown(
                id='locations',
                options= [{'label':loc, 'value':loc} for loc in df['Location'].dropna().unique()],
                multi= True,
                searchable = True,
                placeholder = 'Select multiple : regions'),
        dbc.Col(width=8)]),
    ],style={"background-color": colors['background'], "border": "solid", "padding": "5px"}),
    dbc.Row([
        dbc.Col(html.Div([
            dbc.Row([
                dbc.Col(html.Div([dcc.Graph(id='Indicateur-1')])),
                dbc.Col(html.Div([dcc.Graph(id='Indicateur-2')]))
        ]),
            dbc.Row([html.Div([dcc.Graph(id='Barplot-1')])]),
        ]), width=4, style={'margin-right': '150px'}),
        dbc.Col(html.Div([
            dbc.Row(html.Div([dcc.Graph(id='Graphique-1')]),style={'margin-left': '20px'}),
            dbc.Row(html.Div(
        children=[
            html.H5("Table des 100 dernières ventes", style={'text-align': 'right',}),
            dash_table.DataTable(id='fig-5', page_size=10)
        ],
        style={'width': '50%', 'text-align': 'right'}
    )),
        ]), width=6), 
    ]),
], fluid = True)

@callback(
    Output('Indicateur-1', 'figure'),
    [Input('locations', 'value')]
)
def CA_par_mois(locations):
    df_temps = df[df['Location'].isin(locations)] if locations else df
    return plot_chiffre_affaire_mois(df_temps)

@callback(
    Output('Indicateur-2', 'figure'),
    [Input('locations', 'value')]
)
def Vente_par_mois(locations):
    df_temps = df[df['Location'].isin(locations)] if locations else df
    return plot_vente_mois(df_temps)

@callback(
    Output('Barplot-1', 'figure'),
    [Input('locations', 'value')]
)
def Top_10_ventes_par_mois(locations):
    df_temps = df[df['Location'].isin(locations)] if locations else df
    return barplot_top_10_ventes(df_temps)

@callback(
    Output('Graphique-1', 'figure'),
    [Input('locations', 'value')]
)
def Evolution_CA_mois(locations):
    df_temps = df[df['Location'].isin(locations)] if locations else df
    return plot_evolution_chiffre_affaire(df_temps)

@callback(
    Output('fig-5', 'data'),
    [Input('locations', 'value')]
)
def update_table(locations):
    df_temps = df[df['Location'].isin(locations)] if locations else df
    df_temps = df_temps.sort_values(by='Transaction_Date', ascending=False).head(100)
    columns_to_display = ['CustomerID', 'Gender', 'Location', 'Product_Category', 'Quantity', 'Avg_Price', 'Transaction_Date', 'Month', 'Discount_pct']
    return df_temps[columns_to_display].to_dict('records')



if __name__ == '__main__':
    app.run_server(debug=True, port=8053, jupyter_mode="external")

Dash app running on http://127.0.0.1:8053/
